# CS172 Computer Vision I: CAPTCHA Recognition

**Name:** TODO  
**Student ID:** TODO

This notebook is the report for your implementation and experiments. Keep reusable implementation in your package and scripts. The helper functions are provided for convenience; you may modify, replace, or remove any of them to match your implementation and report, unless the cell explicitly states otherwise.

## Report checklist

- Replace every `TODO` with your own code, result, or analysis.
- Modify any helper function when its assumptions do not match your implementation.
- Execute all cells from a clean kernel and keep the outputs visible.
- Keep test data separate from model selection and hyperparameter tuning.
- Give every figure labeled axes, a title or caption, and a short interpretation.
- Export the completed notebook to PDF before submission.

## 0. Setup

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision.io import ImageReadMode, decode_image
from torchvision.transforms import v2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

## 1. Data generation

Demonstrate valid five-digit CAPTCHAs with random colors, positions, rotations, interference lines, pixel noise, and a border.

In [ ]:
# TODO: import your generator without reimplementing it here.
# TODO: create representative (image, label) pairs. Each image must be an
# RGB NumPy uint8 array [H, W, 3]. For a PIL image, use np.asarray(image).
generated_samples = []

In [ ]:
def show_captcha_samples(
    samples: list[tuple[np.ndarray, list[int] | str]], columns=4, title=None
):
    rows = math.ceil(len(samples) / columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(4 * columns, 2.5 * rows), squeeze=False
    )
    for axis in axes.flat:
        axis.axis("off")
    for axis, (image, label) in zip(axes.flat, samples):
        assert isinstance(image, np.ndarray)
        assert image.dtype == np.uint8 and image.ndim == 3 and image.shape[2] == 3
        axis.imshow(image)
        axis.set_title("label: " + "".join(str(value) for value in label))
        axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_captcha_samples(generated_samples, title="Generated CAPTCHA samples")

## 2. Dataset and preprocessing

Verify the custom dataset, training/validation split, preprocessing, and augmentation behavior. Images must be RGB `torch.Tensor` objects shaped `[3, H, W]`; both `torch.uint8` and `torch.float32` are supported below. Floating-point images should be in `[0, 1]` for display. If your dataset returns normalized tensors, you may need to unnormalize them for visualization.

In [ ]:
# TODO: import and construct separate training and
# validation datasets. Apply random augmentation only to training data.
train_dataset = None
val_dataset = None

In [ ]:
def inspect_dataset_sample(dataset: Dataset, name: str):
    image, label = dataset[0]
    print(f"{name} size: {len(dataset):,}")
    print(f"{name}[0] image: shape={tuple(image.shape)}, dtype={image.dtype}")
    label_text = "".join(str(int(value)) for value in label)
    print(
        f"{name}[0] label: shape={tuple(label.shape)}, dtype={label.dtype}, value={label_text}"
    )
    assert isinstance(image, torch.Tensor), "expected a torch.Tensor image"
    assert image.ndim == 3 and image.shape[0] == 3, "expected an RGB [3, H, W] image"
    assert image.dtype in (
        torch.uint8,
        torch.float32,
    ), "expected uint8 or float32 image data"
    assert tuple(label.shape) == (5,), "expected a five-digit label"
    assert label.dtype == torch.long, "expected torch.long labels"


inspect_dataset_sample(train_dataset, "training dataset")
inspect_dataset_sample(val_dataset, "validation dataset")


def show_dataset_indices(dataset: Dataset, indices: list[int], title: str, columns=4):
    rows = math.ceil(len(indices) / columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(4 * columns, 2.5 * rows), squeeze=False
    )
    for axis, index in zip(axes.flat, indices):
        image, label = dataset[index]
        assert isinstance(image, torch.Tensor)
        assert image.ndim == 3 and image.shape[0] == 3
        assert image.dtype in (torch.uint8, torch.float32)
        image = image.detach().cpu()
        if image.dtype == torch.uint8:
            image = v2.ToDtype(torch.float32, scale=True)(image)

        # TODO (optional): undo channel normalization for visualization.
        # mean = torch.tensor(YOUR_MEAN, dtype=image.dtype).view(3, 1, 1)
        # std = torch.tensor(YOUR_STD, dtype=image.dtype).view(3, 1, 1)
        # image = image * std + mean

        assert image.min().item() >= 0.0 and image.max().item() <= 1.0
        label_text = "".join(str(int(value)) for value in label)
        axis.imshow(image.permute(1, 2, 0))
        axis.set_title(f"label: {label_text}")
        axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


# Repeating index 0 should reveal random training augmentation, if enabled.
show_dataset_indices(
    train_dataset, [0] * 8, "Repeated draws of one augmented training sample"
)

# Distinct validation samples should use deterministic preprocessing only.
val_indices = list(range(min(8, len(val_dataset)))) if val_dataset is not None else []
show_dataset_indices(
    val_dataset, val_indices, "Validation samples without random augmentation"
)

## 3. Model

Verify the interface `[batch, 3, height, width] → [batch, 10 classes, 5 digits]`, and report parameters as well as computational cost. The profiler below estimates FLOPs only for operators supported by PyTorch's profiler, so state the input size and treat the result as an approximation.

In [ ]:
# TODO: import and instantiate one of your models.
model = None

MODEL_CHECK_BATCH_SIZE = 4
MODEL_INPUT_SIZE = (80, 260)  # TODO: (height, width) expected by your model
MODEL_INPUT_DTYPE = torch.uint8  # TODO: change if your model expects floating-point input

In [ ]:
height, width = MODEL_INPUT_SIZE
if MODEL_INPUT_DTYPE == torch.uint8:
    random_images = torch.randint(0, 256, (MODEL_CHECK_BATCH_SIZE, 3, height, width), dtype=torch.uint8)
elif MODEL_INPUT_DTYPE == torch.float32:
    random_images = torch.rand(MODEL_CHECK_BATCH_SIZE, 3, height, width, dtype=torch.float32)
else:
    raise ValueError(f"Unsupported model input dtype: {MODEL_INPUT_DTYPE}")
random_labels = torch.randint(0, 10, (MODEL_CHECK_BATCH_SIZE, 5), dtype=torch.long)

print(f"random images: {tuple(random_images.shape)}, {random_images.dtype}")
print(f"random labels: {tuple(random_labels.shape)}, {random_labels.dtype}")

model_device = next(model.parameters()).device
model.eval()
with torch.inference_mode():
    random_logits = model(random_images.to(model_device))
print(f"model output: {tuple(random_logits.shape)}, {random_logits.dtype}")
assert tuple(random_logits.shape) == (MODEL_CHECK_BATCH_SIZE, 10, 5)

In [ ]:
def compute_num_params_and_flops(model, example_images):
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    model_device = next(model.parameters()).device
    activities = [torch.profiler.ProfilerActivity.CPU]
    if model_device.type == "cuda":
        activities.append(torch.profiler.ProfilerActivity.CUDA)
    model.eval()
    with torch.inference_mode(), torch.profiler.profile(activities=activities, with_flops=True) as profile:
        model(example_images[:1].to(model_device))
    estimated_flops = sum(event.flops or 0 for event in profile.key_averages())
    return {
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "estimated_flops_per_image": estimated_flops,
    }

if model is not None:
    cost = compute_num_params_and_flops(model, random_images)
    for name, value in cost.items():
        print(f"{name}: {value:,}")

## 4. Training report

Select the best checkpoint using validation full-image accuracy, never test performance. Plot training and validation loss, per-digit accuracy, and full-image accuracy over epochs.

### Final training configuration

| Setting | Value |
|---|---|
| Seed | TODO |
| Model | TODO |
| Training images | TODO |
| Optimizer | TODO |
| Batch size | TODO |
| Initial learning rate and schedule | TODO |
| Weight decay | TODO |
| Data augmentation | TODO |
| Epochs and best-checkpoint epoch | TODO |

### TensorBoard curve reader

The functions below read scalar summaries directly from a TensorBoard log directory. The tag mapping must match the names recorded by your training pipeline. TensorBoard is optional and can be installed with `pip install tensorboard`. 

Leveraging Tensorboard to plot curves is just an example. **You can save the metrics in any format you prefer, as long as you can plot them.**

In [ ]:
def read_tensorboard_scalars(log_dir):
    try:
        from tensorboard.backend.event_processing.event_accumulator import (
            EventAccumulator,
        )
    except ImportError as error:
        raise ImportError("Install TensorBoard to read event files.") from error

    accumulator = EventAccumulator(str(log_dir))
    accumulator.Reload()
    history = {}
    for tag in accumulator.Tags().get("scalars", []):
        events = accumulator.Scalars(tag)
        history[tag] = {
            "step": [event.step for event in events],
            "value": [event.value for event in events],
        }
    return history


def plot_training_curves(history, tag_map):
    metrics = ("loss", "digit_acc", "image_acc")
    titles = ("Loss", "Per-digit accuracy", "Full-image accuracy")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    for axis, metric, title in zip(axes, metrics, titles):
        for split in ("train", "val"):
            tag = tag_map[f"{split}_{metric}"]
            series = history[tag]
            axis.plot(series["step"], series["value"], label=split)
        axis.set(xlabel="epoch", title=title)
        axis.grid(alpha=0.25)
        axis.legend()
    plt.show()

In [ ]:
# TODO: point to the selected run and map the six required curves to the
# scalar tags recorded by your training pipeline.
TENSORBOARD_LOG_DIR = None  # TODO: selected run's TensorBoard directory
TENSORBOARD_TAGS = {
    "train_loss": "train/loss",
    "val_loss": "val/loss",
    "train_digit_acc": "train/digit_acc",
    "val_digit_acc": "val/digit_acc",
    "train_image_acc": "train/image_acc",
    "val_image_acc": "val/image_acc",
}

if TENSORBOARD_LOG_DIR is not None:
    final_history = read_tensorboard_scalars(TENSORBOARD_LOG_DIR)
    print("Available scalar tags:", sorted(final_history))
    plot_training_curves(final_history, TENSORBOARD_TAGS)

### TODO: Training analysis

Discuss convergence, stability, the best-checkpoint epoch, and the train–validation gap.

## 5. Performance evaluation

Evaluate the checkpoint selected by validation full-image accuracy exactly once on the five numeric test sets under `data/test/` (500 images in total). Concatenate the test sets and report aggregate per-digit and full-image accuracy.

In [ ]:
# TODO: load the checkpoint selected by validation performance.
test_model = None

# TODO: provide the deterministic preprocessing used during validation.
# Use None only when preprocessing is performed inside the model.
test_transform = None

TEST_DIRS = [
    Path("data/test/test0"),
    Path("data/test/test1"),
    Path("data/test/test2"),
    Path("data/test/test3"),
    Path("data/test/test4"),
]
EVAL_BATCH_SIZE = 64  # TODO: choose an inference batch size

### Evaluation helpers

This small evaluation-only `Dataset` loads RGB images, parses labels from filenames, and applies deterministic preprocessing. The five datasets are concatenated below so the reported metrics cover all 500 test images.

In [ ]:
# CAUTION: DO NOT modify the code below. It will be used to evaluate your model on the hidden test set.

class EvaluationDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_paths = sorted(
            path for path in Path(image_dir).iterdir()
            if path.suffix.lower() in {".png", ".jpg", ".jpeg"}
        )
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        image = decode_image(str(image_path), mode=ImageReadMode.RGB)
        label_text = image_path.stem.split("#", maxsplit=1)[0]
        label = torch.tensor([int(digit) for digit in label_text])
        if self.transform is not None:
            image = self.transform(image)
        return image, label

@torch.inference_mode()
def evaluate_accuracy(model: torch.nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    correct_digits = 0
    correct_images = 0
    num_digits = 0
    num_images = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        logits = model(images)
        assert logits.shape == (targets.shape[0], 10, 5)
        predictions = logits.argmax(dim=1)
        correct = predictions.eq(targets)
        correct_digits += correct.sum().item()
        correct_images += correct.all(dim=1).sum().item()
        num_digits += targets.numel()
        num_images += targets.shape[0]

    digit_accuracy = correct_digits / num_digits
    image_accuracy = correct_images / num_images
    score = (
        20 * max(digit_accuracy - 0.95, 0.0) / 0.05
        + 20 * max(image_accuracy - 0.90, 0.0) / 0.10
    )
    return {
        "num_images": num_images,
        "digit_acc": digit_accuracy,
        "image_acc": image_accuracy,
        "score": score,
    }

In [ ]:
test_datasets = {
    path.name: EvaluationDataset(path, test_transform) for path in TEST_DIRS
}
combined_test_dataset = ConcatDataset(list(test_datasets.values()))
assert len(combined_test_dataset) == 500
test_loader = DataLoader(
    combined_test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False
)

# Run the final evaluation after connecting the selected model and preprocessing.
test_model = test_model.to(DEVICE)
test_result = evaluate_accuracy(test_model, test_loader, DEVICE)
print(f"images:              {test_result['num_images']}")
print(f"per-digit accuracy:  {test_result['digit_acc']:.2%}")
print(f"full-image accuracy: {test_result['image_acc']:.2%}")
print(f"evaluation score:    {test_result['score']:.2f}")

## 6. Analysis and ablations

For every comparison, change only the named factor and keep unrelated settings fixed. Select each run's checkpoint using validation full-image accuracy. When showing curves over epochs, plot validation metrics. When comparing final performance, report the aggregate test results directly. Use simple plots or tables and state a concise conclusion supported by the results.

### 6.1 Training-set size

Train with at least three training-set sizes while keeping the other settings fixed.

**Results.** TODO: Plot (or report with tables) aggregate per-digit and full-image test accuracy against the number of training images. Report the controlled settings.

**Conclusion.** TODO: Explain how performance scales and whether gains saturate.

### 6.2 Data augmentation and overfitting

Compare matched runs with and without data augmentation.

**Results.** TODO: Plot validation full-image accuracy over epochs for both runs. Report training full-image accuracy at each selected checkpoint and compare aggregate per-digit and full-image test accuracy directly.

**Conclusion.** TODO: Explain how augmentation affects validation behavior, test performance, and overfitting.

### 6.3 Regularization and overfitting

Compare multiple weight-decay values while keeping the model and all other training settings, including augmentation, fixed.

**Results.** TODO: Plot validation full-image accuracy over epochs. Report training full-image accuracy at each selected checkpoint and compare aggregate per-digit and full-image test accuracy for the tested values. Report the controlled settings.

**Conclusion.** TODO: Explain how weight decay affects validation behavior, test performance, and overfitting.

### 6.4 Batch size

Compare multiple batch sizes while keeping the model, data, optimizer, and learning-rate schedule fixed. Measure training time consistently on the same hardware.

**Results.** TODO: Plot validation full-image accuracy over epochs and compare aggregate per-digit and full-image test accuracy and training time.

**Conclusion.** TODO: Discuss accuracy, speed, and optimization trade-offs.

### 6.5 Learning rate

Compare multiple initial learning rates while keeping the optimizer and learning-rate schedule otherwise fixed.

**Results.** TODO: Plot validation full-image accuracy over epochs and compare aggregate per-digit and full-image test accuracy for the tested learning rates. Report the controlled settings.

**Conclusion.** TODO: Discuss convergence speed, stability, and test performance.

### 6.6 Backbone comparison

Compare at least two backbones (e.g., ResNet-18 and ResNet-34) under comparable settings.

**Results.** TODO: Compare parameter counts, computation cost for the same input shape, and aggregate per-digit and full-image test accuracy. Report the controlled settings.

**Conclusion.** TODO: Explain whether additional capacity or depth was useful.

### 6.7 Global average pooling

Compare global average pooling with flattening the final spatial feature map while keeping the backbone and training settings comparable.

**Results.** TODO: Compare parameter counts, training full-image accuracy at the selected checkpoints, and aggregate per-digit and full-image test accuracy. Report the controlled settings.

**Conclusion.** TODO: Analyze spatial information, parameter cost, and overfitting risk.

### 6.8 Prediction visualization

Show a small set of test predictions beside their ground-truth labels. If full-image accuracy is below 100%, include and analyze failure cases.

**Failure analysis (if exists).** TODO: Identify recurring patterns such as rotation, occlusion, overlap, low contrast, or digit confusions.